In [2]:
import ollama
from langchain_ollama import OllamaLLM
model_name = "hf.co/bartowski/gemma-2-2b-it-GGUF:Q4_K_M"

llm = OllamaLLM(
    model=model_name,
    temperature=0.2,
    num_predict=256,
)

c:\Users\harip\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Human Prompt

In [3]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

print(llm.invoke("who is the man's best friend?"))

The most common answer to "Who is a man's best friend?" is **a dog**. 

Here's why:

* **Historical and Cultural Significance:** Dogs have been companions to humans for thousands of years, playing roles in hunting, herding, guarding, and even companionship. This long-standing bond makes dogs deeply ingrained in human culture.
* **Emotional Support:**  Dogs offer unconditional love, loyalty, and a sense of comfort that can be invaluable to humans. They provide emotional support during tough times and bring joy into our lives.
* **Physical Benefits:** Dogs encourage physical activity through walks and playtime, which can improve overall health and well-being. 
* **Social Connection:**  Dogs often act as social catalysts, bringing people together in parks, dog parks, or even just on walks.

While dogs are the most popular choice for "man's best friend," it's important to remember that:

* **Cats can be great companions too!** Many people find cats to be affectionate and independent friend

In [4]:
msg = llm.invoke(
    [
        HumanMessage(content="What month follows June?")
    ]
)

print(msg)

July ☀️ 



Human and System Prompt

In [5]:
msg = llm.invoke([
    SystemMessage(content="You are a helpful AI bot that assists a user in choosing the perfect book to read in one short sentence"), 
    HumanMessage(content="I enjoy mystery novels, what should I read?")
])

print(msg)

Try "The Silent Patient" by Alex Michaelides for a gripping psychological thriller. 



Human, AI and System Prompt

In [6]:
msg = llm.invoke(
    [
        SystemMessage(content="You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
        HumanMessage(content="I like high-intensity workouts, what should I do?"),
        AIMessage(content="You should try a CrossFit class"),
        HumanMessage(content="How often should I attend?")
    ]
)

print(msg)

You could aim for 2-3 times per week to start. 



Prompt Template

In [7]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("Tell me one {adjective} joke about {topic}")
input = {"adjective": "funny", "topic": "cat"}

prompt.invoke(input)

StringPromptValue(text='Tell me one funny joke about cat')

Chat Prompt Template

In [8]:
from langchain_core.prompts import ChatPromptTemplate

# List of tuple as individual chat this will convert the values to SystemMessage and HumanMessage class respectively as shown in the output
prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a helpful assitant"),
    ("user", "Tell me a joke about {topic}")
])

input = {"topic" : "cat"}

prompt.invoke(input)

ChatPromptValue(messages=[SystemMessage(content='you are a helpful assitant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me a joke about cat', additional_kwargs={}, response_metadata={})])

Message Placeholder

In [9]:
from langchain_core.prompts import MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a helpful assistant"),
    MessagesPlaceholder("msgs")
])

input = {"msgs" : [HumanMessage(content="What is the day after Tuesday?")]}

prompt.invoke(input)

ChatPromptValue(messages=[SystemMessage(content='you are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the day after Tuesday?', additional_kwargs={}, response_metadata={})])

In [10]:
chain = prompt | llm
response =chain.invoke(input)
print(response)

The day after Tuesday is **Wednesday**. 😊 



Output Parser

In [11]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class Joke(BaseModel):
    setup: str = Field(description="Question to set up a joke")
    punchline: str = Field(description="Answer to resolve the joke")


In [12]:
joke_query = "Tell me a joke"

output_parser = JsonOutputParser(pydantic_object=Joke)

format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions" : format_instructions}
)

chain = prompt | llm | output_parser

response = chain.invoke({"query": joke_query})
print(response)

{'setup': "Why don't scientists trust atoms?", 'punchline': 'Because they make up everything!'}


In [13]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser() # since this is a csv format ouput no need an object for that  

output_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\nList five {subject}\n",
    input_variables=["subject"],
    partial_variables={"format_instructions" : output_instructions}
)

print(prompt.invoke({"subject": "ice cream flavours"}))

chain = prompt | llm | output_parser

response = chain.invoke({"subject": "ice cream flavours"})
print(response)

text='Answer the user query.\nYour response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`\nList five ice cream flavours\n'
['chocolate', 'vanilla', 'strawberry', 'mint chocolate chip', 'cookie dough ']


In [14]:
"""
    Import the necessary components to create a JSON output parser.
    Create a prompt template that requests information in JSON format (hint: use the provided template).
    Build a chain that connects your prompt, LLM, and JSON parser.
    Test your parser using at least three different inputs.
    Access and display specific fields from the parsed JSON output.
    Verify that your output is properly structured and accessible as a Python dictionary.
"""

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="title of the movie")
    director: str = Field(description="Director of the movie")
    year: str = Field(description="Movie release year")
    genre: str = Field(description="Genre of the movie")

# Create your JSON parser
json_parser = JsonOutputParser(pydantic_object=Movie)

# Create the format instructions
format_instructions = """RESPONSE FORMAT: Return ONLY a single JSON object—no markdown, no examples, no extra keys.  It must look exactly like:
{
  "title": "movie title",
  "director": "director name",
  "year": 2000,
  "genre": "movie genre"
}

IMPORTANT: Your response must be *only* that JSON.  Do NOT include any illustrative or example JSON."""

# Create prompt template with instructions
prompt_template = PromptTemplate(
    template="""You are a JSON-only assistant.

Task: Generate info about the movie "{movie_name}" in JSON format.

{format_instructions}
""",
    input_variables=["movie_name"],
    partial_variables={"format_instructions": format_instructions},
)

# Create the chain
movie_chain = prompt_template | llm | json_parser

# Test with a movie name
movie_name = "The Matrix"
result = movie_chain.invoke({"movie_name": movie_name})

# Print the structured result
print("Parsed result:")
print(f"Title: {result['title']}")
print(f"Director: {result['director']}")
print(f"Year: {result['year']}")
print(f"Genre: {result['genre']}")

Parsed result:
Title: The Matrix
Director: Lana Wachowski, Lilly Wachowski
Year: 1999
Genre: Science Fiction, Action


Document

In [15]:
from langchain_core.documents import Document

doc = Document(page_content="""Python is an interpreted high-level general-purpose programming language.
 Python's design philosophy emphasizes code readability with its notable use of significant indentation.
""", 
 metadata={
     'my_document_id' : 234234,                     
    'my_document_source' : "About Python",          
    'my_document_create_time' : 1680013019  
 }
)

print(doc)

page_content='Python is an interpreted high-level general-purpose programming language.
 Python's design philosophy emphasizes code readability with its notable use of significant indentation.
' metadata={'my_document_id': 234234, 'my_document_source': 'About Python', 'my_document_create_time': 1680013019}


Document Loaders

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./support/langchain_paper.pdf")

doc = loader.load()

print(doc)

[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': './langchain_paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content="* corresponding author - jkim72@kent.edu \nRevolutionizing Mental Health Care through \nLangChain: A Journey with a Large Language \nModel\nAditi Singh \n Computer Science  \n Cleveland State University  \n a.singh22@csuohio.edu \nAbul Ehtesham  \nThe Davey Tree Expert \nCompany  \nabul.ehtesham@davey.com \nSaifuddin Mahmud  \nComputer Science & \nInformation Systems  \n Bradley University  \nsmahmud@bradley.edu  \nJong-Hoon Kim* \n Computer Science,  \nKent State University,  \njkim72@kent.edu \nAbstract— Mental health challenges are on the rise in our \nmodern society, and the imperative to address mental disorders, \nespecially regarding anxiety, depression, and suicidal thoughts, \nunderscore

In [17]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")

web_doc = loader.load()

print(web_doc)

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://python.langchain.com/v0.2/docs/introduction/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain is an open source framework with a prebuilt agent architecture and integrations for any model or tool—so you can build agents that adapt as fast as the ecosystem evolves', 'language': 'en'}, page_content='LangChain overview - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Conte

Character Splitters

In [18]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size =200, chunk_overlap= 20, separator="\n")

chunks = text_splitter.split_documents(doc) # this will return list of Documents as individual chunks

print(chunks)

[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': './langchain_paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='* corresponding author - jkim72@kent.edu \nRevolutionizing Mental Health Care through \nLangChain: A Journey with a Large Language \nModel\nAditi Singh \n Computer Science  \n Cleveland State University'), Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': './langchain_paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='a.singh22@csuohio.edu \nAbul Ehtesham  \nThe Davey Tree Expert \nCompany  \nabul.ehtesham@davey.com \nSaifuddin Mahmud  \nComputer Science & \nInformation Systems  \n Bradley University'), Document(metadat

In [ ]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# Load the LangChain paper
paper_url = "./support/langchain_paper.pdf"
pdf_loader = PyPDFLoader(paper_url)
pdf_document = pdf_loader.load()

# Load content from LangChain website
web_url = "https://python.langchain.com/v0.2/docs/introduction/"
web_loader = WebBaseLoader(web_url)
web_document = web_loader.load()

# Create two different text splitters
splitter_1 = CharacterTextSplitter(chunk_size=300, chunk_overlap=30, separator="\n")
splitter_2 = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")

# Apply both splitters to the PDF document
chunks_1 = splitter_1.split_documents(pdf_document)
chunks_2 = splitter_2.split_documents(pdf_document)

# Define a function to display document statistics
def display_document_stats(docs, name):
    """Display statistics about a list of document chunks"""
    total_chunks = len(docs)
    total_chars = sum(len(doc.page_content) for doc in docs)
    avg_chunk_size = total_chars / total_chunks if total_chunks > 0 else 0
    
    # Count unique metadata keys across all documents
    all_metadata_keys = set()
    for doc in docs:
        all_metadata_keys.update(doc.metadata.keys())
    
    # Print the statistics
    print(f"\n=== {name} Statistics ===")
    print(f"Total number of chunks: {total_chunks}")
    print(f"Average chunk size: {avg_chunk_size:.2f} characters")
    print(f"Metadata keys preserved: {', '.join(all_metadata_keys)}")
    
    if docs:
        print("\nExample chunk:")
        example_doc = docs[min(5, total_chunks-1)]  # Get the 5th chunk or the last one if fewer
        print(f"Content (first 150 chars): {example_doc.page_content[:150]}...")
        print(f"Metadata: {example_doc.metadata}")
        
        # Calculate length distribution
        lengths = [len(doc.page_content) for doc in docs]
        min_len = min(lengths)
        max_len = max(lengths)
        print(f"Min chunk size: {min_len} characters")
        print(f"Max chunk size: {max_len} characters")

# Display stats for both chunk sets
display_document_stats(chunks_1, "Splitter 1")
display_document_stats(chunks_2, "Splitter 2")


=== Splitter 1 Statistics ===
Total number of chunks: 95
Average chunk size: 263.80 characters
Metadata keys preserved: moddate, producer, creationdate, title, creator, source, page, author, page_label, total_pages

Example chunk:
Content (first 150 chars): comprehensive support within the field of mental health. 
Additionally, the paper discusses the implementation of 
Streamlit to enhance the user ex pe...
Metadata: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'title': 's8329 final', 'source': './langchain_paper.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}
Min chunk size: 49 characters
Max chunk size: 299 characters

=== Splitter 2 Statistics ===
Total number of chunks: 147
Average chunk size: 170.55 characters
Metadata keys preserved: moddate, producer, creationdate, title, creator, source, page, author, page_label, total_pages

Example chunk:
Content (first 150 cha

Embedding Model

In [20]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="hf.co/mixedbread-ai/mxbai-embed-large-v1:F16"
)

texts = [text.page_content for text in chunks]

query_embed = embeddings.embed_documents(texts)

print(query_embed)

[[0.009011985, -0.023451429, -0.020986753, 0.032029696, -0.013120909, -0.048690245, 0.026213361, -0.014071218, -0.012069448, 0.056731354, 0.024830779, 0.016476246, -0.007468245, -0.039673593, -0.021992614, -0.0018170808, -0.03335529, -0.03203159, -0.029010257, 0.031866338, 0.012422734, 0.024978071, -0.10448621, 0.0017920588, -0.039300933, 0.054641858, 0.039333224, -0.0068915426, 0.06620624, 0.045576908, 0.015411759, -0.0055737724, 0.029667625, -0.019592654, -0.045484763, -0.030578816, 0.024489805, -0.050412226, 0.00037978226, 0.0029262945, -0.005863086, -0.031338863, 0.040563304, -0.06740257, -0.112299286, -0.0069537335, 0.038936414, -0.020167453, 0.014503935, -0.033978518, -0.022660296, 0.007977128, 0.008607119, 0.016212717, 0.008181768, -0.02242287, -0.00080606854, 0.013803927, -0.05021074, 0.005772831, 0.019456606, 0.009513808, 0.002114635, -0.0870844, 0.0047173137, 0.044832237, -0.003066396, -0.012306233, -0.017780501, -0.028134895, -0.052301537, -0.014068223, -0.027364394, -0.0516

Vector Stores

In [21]:
from langchain_chroma import Chroma

doc_search = Chroma.from_documents(chunks, embeddings)

doc = doc_search.similarity_search("Langchain")

print(doc)

[Document(id='c29473ac-241d-4505-8eb3-b9cdda3fb7e5', metadata={'total_pages': 6, 'author': 'IEEE', 'page_label': '1', 'producer': 'PyPDF', 'creator': 'Microsoft Word', 'moddate': '2023-12-31T03:52:06+00:00', 'source': './langchain_paper.pdf', 'creationdate': '2023-12-31T03:50:13+00:00', 'title': 's8329 final', 'page': 0}, page_content='II. LANGCHAIN \nLangChain, with its open -source essence, emerges as a \npromising solution, aiming to simplify the complex process of \ndeveloping applications powered by large language models'), Document(id='58bff6c6-0fac-4240-8c9f-ba430bd1b25d', metadata={'producer': 'PyPDF', 'page_label': '2', 'creator': 'Microsoft Word', 'creationdate': '2023-12-31T03:50:13+00:00', 'moddate': '2023-12-31T03:52:06+00:00', 'source': './langchain_paper.pdf', 'title': 's8329 final', 'page': 1, 'total_pages': 6, 'author': 'IEEE'}, page_content='applications. Whether your desire is to unlock deeper natural \nlanguage understanding , enhance data, or circumvent \nlanguage 

Retrivers

In [22]:
retriver = doc_search.as_retriever()

docs = retriver.invoke("Lanchain")

print(docs)

[Document(id='c29473ac-241d-4505-8eb3-b9cdda3fb7e5', metadata={'total_pages': 6, 'author': 'IEEE', 'source': './langchain_paper.pdf', 'page': 0, 'moddate': '2023-12-31T03:52:06+00:00', 'creationdate': '2023-12-31T03:50:13+00:00', 'creator': 'Microsoft Word', 'producer': 'PyPDF', 'title': 's8329 final', 'page_label': '1'}, page_content='II. LANGCHAIN \nLangChain, with its open -source essence, emerges as a \npromising solution, aiming to simplify the complex process of \ndeveloping applications powered by large language models'), Document(id='de97031b-1c27-4201-ad58-b485e1d0bb51', metadata={'source': './langchain_paper.pdf', 'producer': 'PyPDF', 'title': 's8329 final', 'creator': 'Microsoft Word', 'page_label': '2', 'page': 1, 'author': 'IEEE', 'moddate': '2023-12-31T03:52:06+00:00', 'creationdate': '2023-12-31T03:50:13+00:00', 'total_pages': 6}, page_content='Off-the-Shelf Chains: LangChain offers pre -configured \nchains, which are structured assemblies of components \ntailored to acc

In [37]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

parent_splitter = CharacterTextSplitter(chunk_size = 2000, chunk_overlap = 20, separator="\n")
child_splitter = CharacterTextSplitter(chunk_size = 400, chunk_overlap = 20, separator="\n")

vector_store = Chroma(
    collection_name="split_parents", embedding_function=embeddings
)

store = InMemoryStore()

retriver = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

In [32]:
retriver.add_documents(doc) # this is the pdf file loaded 

print(len(list(store.yield_keys())))

4


In [34]:
sub_doc = vector_store.similarity_search("Langchain")
print(sub_doc[0].page_content)

II. LANGCHAIN


In [35]:
retrived_docs = retriver.invoke("Lanchain")
print(retrived_docs[0].page_content)

II. LANGCHAIN 
LangChain, with its open -source essence, emerges as a 
promising solution, aiming to simplify the complex process of 
developing applications powered by large language models


In [44]:
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# 1. Load a document about AI
loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/") # This document was quiet big for the model i was using in my local.
documents = loader.load()

# 2. Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 250, chunk_overlap = 25)
chunks = text_splitter.split_documents(documents)

# 3. Set up embedding model
embedding_model = OllamaEmbeddings(model="hf.co/mixedbread-ai/mxbai-embed-large-v1:F16")

# 4. Create a vector store
vector_store = Chroma.from_documents(doc, embedding_model) # I changed the document here because of the model contenxt window.

# 5. Create a retriever
retriever = vector_store.as_retriever()

# 6. Define a function to search for relevant information
def search_documents(query, top_k=3):
    """Search for documents relevant to a query"""
    # Use the retriever to get relevant documents
    docs = retriever.invoke(query)
    
    # Limit to top_k if specified
    return docs[:top_k]

# 7. Test with a few queries
test_queries = [
    "What is LangChain?",
    "How do retrievers work?",
    "Why is document splitting important?"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    results = search_documents(query)
    # Print the results
    print(results)



Query: What is LangChain?
[Document(id='c29473ac-241d-4505-8eb3-b9cdda3fb7e5', metadata={'page': 0, 'moddate': '2023-12-31T03:52:06+00:00', 'creator': 'Microsoft Word', 'page_label': '1', 'title': 's8329 final', 'author': 'IEEE', 'total_pages': 6, 'source': './langchain_paper.pdf', 'producer': 'PyPDF', 'creationdate': '2023-12-31T03:50:13+00:00'}, page_content='II. LANGCHAIN \nLangChain, with its open -source essence, emerges as a \npromising solution, aiming to simplify the complex process of \ndeveloping applications powered by large language models'), Document(id='de97031b-1c27-4201-ad58-b485e1d0bb51', metadata={'source': './langchain_paper.pdf', 'creationdate': '2023-12-31T03:50:13+00:00', 'moddate': '2023-12-31T03:52:06+00:00', 'producer': 'PyPDF', 'page_label': '2', 'creator': 'Microsoft Word', 'title': 's8329 final', 'author': 'IEEE', 'total_pages': 6, 'page': 1}, page_content='Off-the-Shelf Chains: LangChain offers pre -configured \nchains, which are structured assemblies of c

Memory

In [47]:
from langchain_community.chat_message_histories.in_memory import ChatMessageHistory

chat = llm

history = ChatMessageHistory()

history.add_ai_message("hi!")

history.add_user_message("what is the capital of France?")

print(history.messages)

[AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what is the capital of France?', additional_kwargs={}, response_metadata={})]


In [48]:
ai_response = chat.invoke(history.messages)
ai_response

'The capital of France is **Paris**. 🇫🇷 😊 \n'

In [49]:
history.add_ai_message(ai_response)
history.messages

[AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='what is the capital of France?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='The capital of France is **Paris**. 🇫🇷 😊 \n', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]